In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from utils._preprocessing import _preprocessing_dataset
import os

In [2]:
# get statistics

subtraj_index = 0

def save_subtraj_statistic_feats():
    folder_path = f"./analysis_results/statistical_relevance/features"

    save_path = "./analysis_results//statistical_relevance/features"
    os.makedirs(save_path, exist_ok=True)

    # Check if output files already exist
    output_files = [f"{save_path}/4_s1.npy", f"{save_path}/4_s2.npy", 
                    f"{save_path}/4_s3.npy", f"{save_path}/4_s4.npy",
                    f"{save_path}/4_s5.npy", f"{save_path}/4_s6.npy",]
    if all(os.path.exists(file) for file in output_files):
        print("Output files already exist. Exiting.")
        return
        
    tag = "test"
    for tag_index in range(5):
        # Load and preprocess features
        features = {f'f{i}': np.load(f"{folder_path}/f{i}_{tag}_{tag_index}.npy", allow_pickle=True) for i in range(1, 6)}
        for key in ['f2', 'f3', 'f5']:
            features[key] = MinMaxScaler().fit_transform(np.abs(features[key]).T).T
    
        # Save combined features
        np.save(f"{save_path}/{tag_index}_s1.npy", features['f1']) # AC
        np.save(f"{save_path}/{tag_index}_s2.npy", features['f1'] * features['f5']) # SG
        np.save(f"{save_path}/{tag_index}_s3.npy", features['f3'] * features['f2']) # VD
        np.save(f"{save_path}/{tag_index}_s4.npy", features['f4'] * features['f2']) # VD

        np.save(f"{save_path}/{tag_index}_s5.npy", features['f3']) # Jump
        np.save(f"{save_path}/{tag_index}_s6.npy", features['f2']) # NG

save_subtraj_statistic_feats()

Output files already exist. Exiting.


In [3]:
from utils._statistical_relevance import _get_statistical_features, array_split


# Sample array with dimensions (2, 1000)

def inject_subtrajectory(trajectory, i, j):
    '''
    inject j-th subtrajectory into i-th subtrajectory (overwrites i-th subtrajectory)
    '''
    # Calculate the start and end indices for both subtrajectories
    array = trajectory.copy()
    additional_window = 0
    start_i = 25 * i - additional_window if 25 * i - additional_window > 0 else 0
    end_i = 225 + 25 * i  + additional_window if 225 + 25 * i + additional_window <= len(array[0]) else len(array[0])
    start_j = 25 * j - additional_window if 25 * j - additional_window > 0 else 0
    end_j = 225 + 25 * j  + additional_window if 225 + 25 * j + additional_window <= len(array[0]) else len(array[0])

    # Extract the original subtrajectories
    subtraj_i = array[:, start_i:end_i].copy()
    subtraj_j = array[:, start_j:end_j].copy()


    # Adjust subtrajectory j to match the initial value of subtraj i
    subtraj_j_adjusted = subtraj_j - subtraj_j[:, 0][:, np.newaxis] + subtraj_i[:, 0][:, np.newaxis]
    initial_gap = subtraj_j[:, 0] - subtraj_i[:, 0]
    last_gap = subtraj_j[:, -1] - subtraj_i[:, -1]

    # Inject the adjusted subtrajectory j into the region of subtrajectory i
    array[:, start_i:end_i] = subtraj_j_adjusted

    # Adjust the rest of the array to match the final value of injected subtrajectory
    if end_i < len(array[0]):
        array[:, end_i:] = array[:, end_i:] + (last_gap - initial_gap)[:, np.newaxis]

    return array

# defined loss as selected stat is NOT similar, other stats are similar


In [4]:

# Vectorized loss computation
def compute_loss(origin_stat, injected_stat, total_stat, stat_name, non_constant_index):
    loss_val = -np.sum((origin_stat[stat_name][:,non_constant_index] - injected_stat[stat_name][:,non_constant_index])**2)
    for stat in total_stat.keys():
        if stat != stat_name:
            loss_val += np.sum((origin_stat[stat][:,non_constant_index] - injected_stat[stat][:,non_constant_index])**2)
    return loss_val

# Optimized function to handle the operations
def anti_relative_loss_subtrajectories(total_stat, stat_name, traj_dataset, inverse=False):

    # Initialize arrays
    inject_trajectory = -np.inf*np.ones_like(traj_dataset)
    loss_total = -np.inf*np.ones_like(total_stat[stat_name])
    best_index = -np.inf*np.ones(len(total_stat[stat_name]), dtype=int)
    loss_min_index = -np.inf*np.ones(len(total_stat[stat_name]), dtype=int)

    injected_total_stat = {
        "AC": -np.inf* np.ones_like(total_stat[stat_name]),
        "CS": -np.inf* np.ones_like(total_stat[stat_name]),
        "SG": -np.inf* np.ones_like(total_stat[stat_name]),
        "VD": -np.inf* np.ones_like(total_stat[stat_name])
    }

    # Iterate over each sample index in the dataset
    for index in tqdm(range(len(total_stat[stat_name]))):
        verbose = False
        origin_stat = {}
        # initially erase the all-constant trajectory
        non_constant_index = np.where(total_stat["AC"][index] != 1)[0]

        if not inverse:
            max_stat_index = np.argmax(total_stat[stat_name][index][non_constant_index])
        else:
            max_stat_index = np.argmin(total_stat[stat_name][index][non_constant_index])

        if len(non_constant_index) == 0:
            # skip for all-constant trajectory
            print("skip all-constant")
            continue

        if len(non_constant_index) != len(total_stat[stat_name][index]):
            # print("Constant subtrajectory detected:", len(non_constant_index), "!=", len(total_stat[stat_name][index]))
            verbose = True

        original_traj = traj_dataset[index].copy()
        origin_stat["AC"], origin_stat["CS"], origin_stat["SG"], origin_stat["VD"] = total_stat["AC"][index][np.newaxis, :], total_stat["CS"][index][np.newaxis, :], total_stat["SG"][index][np.newaxis, :], total_stat["VD"][index][np.newaxis, :]

        loss_subtraj = np.ones_like(total_stat[stat_name][index]) * np.inf
        best_loss_val = np.inf
        best_injected_traj = None
    

        full_injected_traj = []
        valid_indices = []

        for subtraj_index in non_constant_index:
            # Skip subtrajectories near the max_stat_index
            if abs(subtraj_index - max_stat_index) < 225 / 25:
                continue
            
            # Inject subtrajectory
            injected_traj = inject_subtrajectory(original_traj, max_stat_index, subtraj_index)
            full_injected_traj.append(injected_traj)
            valid_indices.append(subtraj_index)

        # Batch process all injected trajectories
        if full_injected_traj:
            splited_trajs = array_split(np.array(full_injected_traj))  # Split all trajectories at once
            all_stats = _get_statistical_features(splited_trajs)  # Get all stats at once
            s1_batch, s2_batch, s3_batch, s4_batch = all_stats
            
            # Compute losses and find best
            best_loss_val = float('inf')
            best_injected_traj = None
            best_idx = None
            
            for i, subtraj_index in enumerate(valid_indices):
                injected_stat = {"AC": s1_batch[i][np.newaxis, :], "CS": s2_batch[i][np.newaxis, :], "SG": s3_batch[i][np.newaxis, :], "VD": s4_batch[i][np.newaxis, :]}
                loss_val = compute_loss(origin_stat, injected_stat, total_stat, stat_name, non_constant_index)
                loss_subtraj[subtraj_index] = loss_val
                
                # Update best values
                if loss_val < best_loss_val:
                    best_loss_val = loss_val
                    best_injected_traj = full_injected_traj[i]
                    best_idx = i
            
            # Update injected_total_stat with best result
            injected_total_stat["AC"][index] = s1_batch[best_idx]
            injected_total_stat["CS"][index] = s2_batch[best_idx]
            injected_total_stat["SG"][index] = s3_batch[best_idx]
            injected_total_stat["VD"][index] = s4_batch[best_idx]

            # Store results
            inject_trajectory[index] = best_injected_traj
            loss_total[index] = loss_subtraj
            best_index[index] = max_stat_index
            loss_min_index[index] = np.argmin(loss_subtraj)
            


    return inject_trajectory, injected_total_stat, loss_total, best_index, loss_min_index


In [5]:

# random injection baseline
def anti_relative_loss_subtrajectories_random(total_stat, stat_name, traj_dataset, inverse=False):

    # Initialize arrays
    inject_trajectory = -np.inf*np.ones_like(traj_dataset)
    loss_total = -np.inf*np.ones_like(total_stat[stat_name])
    best_index = -np.inf*np.ones(len(total_stat[stat_name]), dtype=int)
    loss_min_index = -np.inf*np.ones(len(total_stat[stat_name]), dtype=int)

    injected_total_stat = {
        "AC": -np.inf* np.ones_like(total_stat[stat_name]),
        "CS": -np.inf* np.ones_like(total_stat[stat_name]),
        "SG": -np.inf* np.ones_like(total_stat[stat_name]),
        "VD": -np.inf* np.ones_like(total_stat[stat_name])
    }

    # Iterate over each sample index in the dataset
    for index in tqdm(range(len(total_stat[stat_name]))):
        verbose = False

        origin_stat = {}

        non_constant_index = np.where(total_stat["AC"][index] != 1)[0]

        if not inverse:
            max_stat_index = np.argmax(total_stat[stat_name][index][non_constant_index])
        else:
            max_stat_index = np.argmin(total_stat[stat_name][index][non_constant_index])

        if len(non_constant_index) == 0:
            # skip for all-constant trajectory
            print("skip all-constant")
            continue
 
        if len(non_constant_index) != len(total_stat[stat_name][index]):
            # print("Constant subtrajectory detected:", len(non_constant_index), "!=", len(total_stat[stat_name][index]))
            verbose = True

        original_traj = traj_dataset[index].copy()
        origin_stat["AC"], origin_stat["CS"], origin_stat["SG"], origin_stat["VD"] = total_stat["AC"][index][np.newaxis, :], total_stat["CS"][index][np.newaxis, :], total_stat["SG"][index][np.newaxis, :], total_stat["VD"][index][np.newaxis, :]

        loss_subtraj = np.ones_like(total_stat[stat_name][index]) * np.inf
        best_loss_val = np.inf
        best_injected_traj = None

        # Iterate through the subtrajectories
        # if all non constant index cannot satisfy the condition, skip.
        if not any(abs(subtraj_index - max_stat_index) >= 225 / 25 for subtraj_index in non_constant_index):
            pass
        else:
            # choice under the condition
            candidates = [i for i in non_constant_index if abs(i - max_stat_index)  >= 225 / 25]
            subtraj_index = np.random.choice(candidates)

            # Inject subtrajectory
            injected_traj = inject_subtrajectory(original_traj, max_stat_index, subtraj_index)
            splited_traj = array_split(np.array([injected_traj]))  # Split the injected trajectory
            s1, s2, s3, s4 = _get_statistical_features(splited_traj)
            injected_stat = {"AC": s1, "CS": s2, "SG": s3, "VD": s4}

            # Compute loss
            loss_val = compute_loss(origin_stat, injected_stat, total_stat, stat_name, non_constant_index)
            loss_subtraj[subtraj_index] = loss_val
            # if verbose:
            #     print("Origin stat:", origin_stat)
            #     print("Injected stat:", injected_stat)
            #     print("non constant index:", non_constant_index)

            # Update best values
            if loss_val < best_loss_val:
                best_loss_val = loss_val
                best_injected_traj = injected_traj
                injected_total_stat["AC"][index] = s1
                injected_total_stat["CS"][index] = s2
                injected_total_stat["SG"][index] = s3
                injected_total_stat["VD"][index] = s4

            # Store results
            inject_trajectory[index] = best_injected_traj
            loss_total[index] = loss_subtraj
            best_index[index] = max_stat_index
            loss_min_index[index] = np.argmin(loss_subtraj)


    return inject_trajectory, injected_total_stat, loss_total, best_index, loss_min_index


# Saving

In [6]:
for save_tag_index in [0, 1, 2, 3, 4]:
    print(save_tag_index)
    # if save_tag_index == 0:
    #     continue
        
    traj_dataset = np.load(f"./dataset_noiseless/test_1000/{save_tag_index}.npy", allow_pickle=True)
    traj_dataset, _ = _preprocessing_dataset(traj_dataset)

    
    total_stat_raw = {}
    for index, stat_name in enumerate(["AC","CS", "SG", "VD",]):
        index = index + 1
        print([f"./analysis_results//statistical_relevance/features/{i}_s{index}.npy" for i in [save_tag_index]])
        concat_list = [np.load(f"./analysis_results//statistical_relevance/features/{i}_s{index}.npy") for i in [save_tag_index]]
        total_stat_raw[stat_name] = np.concatenate(concat_list)
    
    stat_names = ["AC", "CS", "SG", "VD"]
    save_path = f"./analysis_results/feature_ablation/injected_dict/{save_tag_index}/"
    os.makedirs(save_path, exist_ok=True)
    print(save_path)
    
    for inverse in [False, True]:
        inv = "_inverse" if inverse else ""
        for stat_name in stat_names:
            if os.path.exists(save_path+f"/{stat_name}_inject_trajectory{inv}.npy"):
                print(f"Skipping {stat_name} with inverse={inverse}")
                
            else:
                inject_trajectory, injected_total_stat, loss_total, best_index, loss_min_index = anti_relative_loss_subtrajectories(total_stat_raw, stat_name, traj_dataset, inverse)
                np.save(save_path + f"/{stat_name}_inject_trajectory{inv}", inject_trajectory)
                np.save(save_path + f"/{stat_name}_injected_total_stat{inv}", injected_total_stat)
                np.save(save_path + f"/{stat_name}_loss_total{inv}", loss_total)
                np.save(save_path + f"/{stat_name}_best_index{inv}", best_index)
                np.save(save_path + f"/{stat_name}_loss_min_index{inv}", loss_min_index)

0
['./analysis_results//statistical_relevance/features/0_s1.npy']
['./analysis_results//statistical_relevance/features/0_s2.npy']
['./analysis_results//statistical_relevance/features/0_s3.npy']
['./analysis_results//statistical_relevance/features/0_s4.npy']
./analysis_results/feature_ablation/injected_dict/0/
Skipping AC with inverse=False
Skipping CS with inverse=False
Skipping SG with inverse=False
Skipping VD with inverse=False
Skipping AC with inverse=True
Skipping CS with inverse=True
Skipping SG with inverse=True
Skipping VD with inverse=True
1
['./analysis_results//statistical_relevance/features/1_s1.npy']
['./analysis_results//statistical_relevance/features/1_s2.npy']
['./analysis_results//statistical_relevance/features/1_s3.npy']
['./analysis_results//statistical_relevance/features/1_s4.npy']
./analysis_results/feature_ablation/injected_dict/1/
Skipping AC with inverse=False
Skipping CS with inverse=False
Skipping SG with inverse=False
Skipping VD with inverse=False
Skipping A

In [7]:
for save_tag_index in [0, 1, 2, 3, 4]:
    print(save_tag_index)

    traj_dataset = np.load(f"./dataset_noiseless/test_1000/{save_tag_index}.npy", allow_pickle=True)
    traj_dataset, _ = _preprocessing_dataset(traj_dataset)

    
    total_stat_raw = {}
    for index, stat_name in enumerate(["AC","CS", "SG", "VD",]):
        index = index + 1
        print([f"./analysis_results//statistical_relevance/features/{i}_s{index}.npy" for i in [save_tag_index]])
        concat_list = [np.load(f"./analysis_results//statistical_relevance/features/{i}_s{index}.npy") for i in [save_tag_index]]
        total_stat_raw[stat_name] = np.concatenate(concat_list)
    
    stat_names = ["AC", "CS", "SG", "VD"]
    save_path = f"./analysis_results/feature_ablation/injected_dict/{save_tag_index}/"
    os.makedirs(save_path, exist_ok=True)
    print(save_path)
    
    for inverse in [False, True]:
        inv = "_inverse" if inverse else ""
        for stat_name in stat_names:
            if os.path.exists(save_path+f"/{stat_name}_inject_trajectory{inv}_rand.npy"):
                print(f"Skipping {stat_name} with inverse={inverse}_rand")
                
            else:
                inject_trajectory, injected_total_stat, loss_total, best_index, loss_min_index = anti_relative_loss_subtrajectories_random(total_stat_raw, stat_name, traj_dataset, inverse)
                np.save(save_path + f"/{stat_name}_inject_trajectory{inv}_rand", inject_trajectory)
                np.save(save_path + f"/{stat_name}_injected_total_stat{inv}_rand", injected_total_stat)
                np.save(save_path + f"/{stat_name}_loss_total{inv}_rand", loss_total)
                np.save(save_path + f"/{stat_name}_best_index{inv}_rand", best_index)
                np.save(save_path + f"/{stat_name}_loss_min_index{inv}_rand", loss_min_index)

0


['./analysis_results//statistical_relevance/features/0_s1.npy']
['./analysis_results//statistical_relevance/features/0_s2.npy']
['./analysis_results//statistical_relevance/features/0_s3.npy']
['./analysis_results//statistical_relevance/features/0_s4.npy']
./analysis_results/feature_ablation/injected_dict/0/
Skipping AC with inverse=False_rand
Skipping CS with inverse=False_rand
Skipping SG with inverse=False_rand
Skipping VD with inverse=False_rand
Skipping AC with inverse=True_rand
Skipping CS with inverse=True_rand
Skipping SG with inverse=True_rand
Skipping VD with inverse=True_rand
1
['./analysis_results//statistical_relevance/features/1_s1.npy']
['./analysis_results//statistical_relevance/features/1_s2.npy']
['./analysis_results//statistical_relevance/features/1_s3.npy']
['./analysis_results//statistical_relevance/features/1_s4.npy']
./analysis_results/feature_ablation/injected_dict/1/
Skipping AC with inverse=False_rand
Skipping CS with inverse=False_rand
Skipping SG with inverse

In [8]:
# save specific anomalous model's injected trajectories (for/inv)
import os

dict_path = f"./analysis_results/feature_ablation/injected_dict"
save_path = f"./analysis_results/feature_ablation/dataset_injected_mse"
os.makedirs(save_path, exist_ok=True)


model_names = ["SubATTM", "SubCTRW", "SubFBM", "SubSBM", "SupFBM", "SupLW", "SupSBM", "STDBM"]
stat_names = ["AC","CS","SG","VD"]
# percentile_index = 1000

output_files = [f"{save_path}/{tag_index}_{model_name}_{stat_name}_inverse.npy" for tag_index in range(5) for model_name in model_names for stat_name in stat_names]
if all(os.path.exists(file) for file in output_files):
    print("Output files already exist. Exiting.")
else:
    for tag_index in tqdm(range(5)):
        traj_dataset = np.load(f"./dataset_noiseless/test_1000/{tag_index}.npy", allow_pickle=True)
        traj_dataset, _ = _preprocessing_dataset(traj_dataset)

        for inverse in [False, True]:
            inv = "_inverse" if inverse else ""
            for stat_name in stat_names:
                try:
                    inject_trajectory =  np.load(dict_path + f"/{tag_index}/{stat_name}_inject_trajectory{inv}.npy", allow_pickle=True)
                    loss_total = np.load(dict_path + f"/{tag_index}/{stat_name}_loss_total{inv}.npy", allow_pickle=True)
                    best_index = np.load(dict_path + f"/{tag_index}/{stat_name}_best_index{inv}.npy", allow_pickle=True)
                    loss_min_index = np.load(dict_path + f"/{tag_index}/{stat_name}_loss_min_index{inv}.npy", allow_pickle=True)

                    for model_index, model_name in enumerate(model_names):
                        # for & inv
                        injected_save_path = f"{save_path}/{tag_index}_{model_name}_{stat_name}{inv}.npy"
                        baseline_save_path = f"{save_path}/{tag_index}_{model_name}_{stat_name}{inv}_base.npy"

                        if os.path.exists(injected_save_path) and os.path.exists(baseline_save_path):
                            pass
                        else:
                            selected_indexes = np.argsort(np.min(loss_total[10000*model_index:10000*(model_index+1)], axis=-1)).flatten()
                            
                            injected_traj_dataset = inject_trajectory[10000*model_index:10000*(model_index+1)]
                            injected_traj_dataset = injected_traj_dataset[selected_indexes]

                            baseline_traj_dataset = traj_dataset[10000*model_index:10000*(model_index+1)]
                            baseline_traj_dataset = baseline_traj_dataset[selected_indexes]

                            
                            np.save(injected_save_path, injected_traj_dataset)
                            np.save(baseline_save_path, baseline_traj_dataset)

                        # random distrub # using prev one
                except Exception as e:
                    # file dose not exist
                    print(e)
        
                

Output files already exist. Exiting.


# Model results under injected trajectories

In [9]:
import torch
from utils._resnet import resnet18_8
from utils._preprocessing import _preprocessing_dataset, _preprocessing_scaling
from utils._erasing_method import _get_dataloader, _evaluate_model
from sklearn.metrics import accuracy_score

In [10]:
model_path = f"./saved_models/resnet18_8_b64_lr0.0001_1000_noiseless/checkpoint.pt"
model = resnet18_8()
model.load_state_dict(torch.load(model_path), strict=False)
model.eval()
print("Done")

Done


In [11]:
model_names = ["SubATTM", "SubCTRW", "SubFBM", "SubSBM", "SupFBM", "SupLW", "SupSBM", "STDBM"]
stat_names = ["AC","CS","SG","VD"]
device = "cuda"
dataset_size = 10000 # max 10000
batch_size = dataset_size//10
for tag_index in [0, 1, 2, 3, 4]:
    disturb_acc_dict = {}
    distrub_res_dict = {}
    save_path = "./analysis_results/feature_ablation/dataset_injected_mse/"
    
    for inverse in [False, True]:
        inv = "_inverse" if inverse else ""
        acc_save_path = f"./analysis_results/feature_ablation/disturb_acc_dict_mse{inv}_{tag_index}.npy"
        res_save_path = f"./analysis_results/feature_ablation/disturb_res_dict_mse{inv}_{tag_index}.npy"
        if os.path.exists(acc_save_path) and os.path.exists(res_save_path):
            print("Output files already exist. Exiting.")
            continue
        else:
            for model_index in (range(len(model_names))):
                disturb_acc_dict[model_names[model_index]] = {}  # Initialize a dict for each label_index
                distrub_res_dict[model_names[model_index]] = {}

                for stat_index in range(len(stat_names)):
                    disturb_acc_dict[model_names[model_index]][stat_names[stat_index]] = {}
                    distrub_res_dict[model_names[model_index]][stat_names[stat_index]] = {}

                    model_name = model_names[model_index]
                    stat_name = stat_names[stat_index]
                    injected_save_path = f"{save_path}/{tag_index}_{model_name}_{stat_name}{inv}.npy"
                    if os.path.exists(injected_save_path):
                        label_traj_dataset = np.load(injected_save_path)
                        
                        X = label_traj_dataset[:dataset_size]
                        y = np.ones(len(X)) * model_index

                        print("current index:", tag_index, model_name, stat_name, inv)

                        
                        X[np.where(X==-np.inf)] = np.nan
                        X = X[~np.isnan(X).any(axis=(1,2))]
                        y = y[~np.isnan(label_traj_dataset).any(axis=(1,2))]

                        # nan processed results
                        print("Nan processed shape:", X.shape, y.shape)

                        plt.plot(X[0][1], label="injected")
                        plt.show()
                        # Get dataloader and evaluate model
                        disturb_ds = _get_dataloader(X, y, batch_size=batch_size)
                        distrub_results, ground_truth = _evaluate_model(model, disturb_ds, device)
                        
                        # Calculate accuracy
                        distrub_acc = accuracy_score(distrub_results, ground_truth)
                        print(distrub_acc)

                        # Store the accuracy in the dictionary
                        disturb_acc_dict[model_names[model_index]][stat_names[stat_index]] = distrub_acc
                        distrub_res_dict[model_names[model_index]][stat_names[stat_index]] = distrub_results

                np.save(acc_save_path, disturb_acc_dict)
                np.save(res_save_path, distrub_res_dict)

    
    ######################### baseline #########################
    disturb_acc_dict = {}
    distrub_res_dict = {}
    save_path = "./analysis_results/feature_ablation/dataset_injected_mse/"
    
    for inverse in [False, True]:
        inv = "_inverse" if inverse else ""
        acc_save_path = f"./analysis_results/feature_ablation/disturb_acc_dict_mse{inv}_base_{tag_index}.npy"
        res_save_path = f"./analysis_results/feature_ablation/disturb_res_dict_mse{inv}_base_{tag_index}.npy"
        if os.path.exists(acc_save_path) and  os.path.exists(res_save_path):
            print("Output files already exist. Exiting.")
            continue
        else:
            for model_index in (range(len(model_names))):
                disturb_acc_dict[model_names[model_index]] = {}  # Initialize a dict for each label_index
                distrub_res_dict[model_names[model_index]] = {}

                for stat_index in range(len(stat_names)):
                    disturb_acc_dict[model_names[model_index]][stat_names[stat_index]] = {}
                    distrub_res_dict[model_names[model_index]][stat_names[stat_index]] = {}

                    model_name = model_names[model_index]
                    stat_name = stat_names[stat_index]
                    baseline_save_path = f"{save_path}/{tag_index}_{model_name}_{stat_name}{inv}_base.npy"
                    print(baseline_save_path)
                    if os.path.exists(baseline_save_path):
                        label_traj_dataset = np.load(baseline_save_path)
                        
                        X = label_traj_dataset[:dataset_size]
                        y = np.ones(len(X)) * model_index

                        # ignore all-constant trajectory (injected trajectory is nan value (-inf))
                        X[np.where(X==-np.inf)] = np.nan
                        X = X[~np.isnan(X).any(axis=(1,2))]
                        y = y[~np.isnan(label_traj_dataset).any(axis=(1,2))]

                        # nan processed results
                        print("Nan processed shape:", X.shape, y.shape)

                        plt.plot(X[0][1], label="injected")
                        plt.show()
                        
    
                        # Get dataloader and evaluate model
                        disturb_ds = _get_dataloader(X, y, batch_size=batch_size)
                        distrub_results, ground_truth = _evaluate_model(model, disturb_ds, device)
                        
                        # Calculate accuracy
                        distrub_acc = accuracy_score(distrub_results, ground_truth)
                        print(distrub_acc)
    
                        # Store the accuracy in the dictionary
                        disturb_acc_dict[model_names[model_index]][stat_names[stat_index]] = distrub_acc
                        distrub_res_dict[model_names[model_index]][stat_names[stat_index]] = distrub_results
    
                np.save(acc_save_path, disturb_acc_dict)
                np.save(res_save_path, distrub_res_dict)

Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.
Output files already exist. Exiting.


In [12]:
"DONE"

'DONE'